# RFW 1:1 Continuous FIQA Threshold Calibration

본 노트북은 사전학습된 얼굴인식 백본(ArcFace, AdaFace, MagFace)을 대상으로,
Product Quantization(PQ $m=128, 64, 32$) 압축 시 발생하는 점수 왜곡에 대해
Continuous FIQA(구간별 선형 basis를 사용하는 스플라인 분위수 회귀와 safety offset) 기반 사후 임계값 보정이 Global Safe 및 FIQA 5-bin 대비 어떤 차이를 만드는지
RFW 공식 4개 인종 그룹(African, Asian, Caucasian, Indian) 10-fold 1:1 검증 프로토콜에서 평가합니다.

### 핵심 계약 및 주의사항 (AGENTS.md 준수)
- **Two-Endpoint Identity Disjointness**: 1:1 impostor pair의 좌/우 identity가 fit과 safety에 걸치지 않도록 양쪽 모두 동일 역할에 배정된 pair만 calibration에 유지합니다.
- **점수 공간 분리**: Cosine similarity `[-1.0, 1.0]`과 PQ ADC negative squared L2 `[-4.0, 0.0]`는 별도 점수 공간으로 독립 보정합니다.
- **PQ 개발 데이터 분리**: PQ 코덱은 LFW View-2 test pair와 분리되며 RFW와의 인물 중복은 미검증(`identity_overlap_verified = False`)된 모델별 development 임베딩(LFW disjoint non-test 1,549장)으로만 학습합니다.
- **대칭 품질 적용**: 1:1 검증에서는 대칭 품질 $q_{\text{pair}} = \min(q_{\text{left}}, q_{\text{right}})$를 적용하여 LFW 1:1 비대칭 DB 검색 모사(query-only)와 분리합니다.
- **보장 주장 금지**: `formal_fmr_guarantee = False`, `fairness_guarantee = False`. 통계적 관측치이며 수학적 보장이 아닙니다.

In [ ]:
# cell 1 : 기본 라이브러리 임포트 및 환경 확인
from __future__ import annotations
from pathlib import Path
import sys
import torch
import pandas as pd

# 프로젝트 루트 경로 설정
PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.rfw_continuous_calibration import run_rfw_calibration_workflow

REQUESTED_DEVICE = "cuda"
if REQUESTED_DEVICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA execution requested but torch.cuda is unavailable; silent CPU fallback is prohibited.")
    device = "cuda"
    print(f"Project root: {PROJECT_ROOT}")
    print(f"PyTorch device: {device} ({torch.cuda.get_device_name(0)})")
else:
    device = "cpu"
    print(f"Project root: {PROJECT_ROOT}")
    print(f"PyTorch device: {device}")


In [ ]:
# cell 2 : 실험 설정 및 파라미터 지정
CONFIG_PATH = PROJECT_ROOT / "configs/experiments/rfw_continuous_calibration.yaml"
OUTPUT_ROOT = PROJECT_ROOT / "results/calibration/rfw_continuous_calibration"

# 실행 모드 선택
EXECUTE = True  # 실제 실행 여부 (False: planned status만 반환)
REAL_DATA = True  # 실제 RFW 바이너리 및 모델 가중치 사용 여부
QUICK_SMOKE = True  # 빠른 축소 검증 여부 (True: African 2 folds 1,200쌍, False: 24,000쌍 전수)
KEEP_RAW_RESULTS = False  # 상세 per-fold/pair 원본 CSV 보존 여부 (False: 요약본/manifest만 보존)
SELECTED_MODEL = "arcface"  # 평가 대상 모델 ('arcface', 'adaface', 'magface')
QUALITY_SOURCE = "cr_fiqa"  # 품질 평가기 ('cr_fiqa': 독립 CR-FIQA, 'embedding_norm': ablation)
SEED = 8972  # 재현성 시드

print(f"Config: {CONFIG_PATH}")
print(f"Execution: execute={EXECUTE}, real_data={REAL_DATA}, quick_smoke={QUICK_SMOKE}, quality_source={QUALITY_SOURCE}")

In [ ]:
# cell 3 : RFW Continuous Calibration 파이프라인 실행
result = run_rfw_calibration_workflow(
    project_root=PROJECT_ROOT,
    config_path=CONFIG_PATH,
    output_root=OUTPUT_ROOT,
    execute=EXECUTE,
    keep_raw_results=KEEP_RAW_RESULTS,
    real_data=REAL_DATA,
    quick_smoke=QUICK_SMOKE,
    selected_model=SELECTED_MODEL,
    quality_source=QUALITY_SOURCE,
    device=device,
    seed=SEED,
)

print(f"Status: {result['status']}")
print(f"Run ID: {result['run_id']}")
print(f"Output Directory: {result['output_dir']}")
print(f"Manifest: {result.get('manifest') is not None}")

In [ ]:
# cell 4 : 결과 요약 표 및 저장 효율 비교 출력
comparison_path = Path(result["comparison_table_path"])
group_path = Path(result["group_summary_path"])

if comparison_path.is_file():
    df_comp = pd.read_csv(comparison_path)
    print("=== Compression vs Calibration Comparison Table ===")
    display(df_comp.head(20))

if group_path.is_file():
    df_grp = pd.read_csv(group_path)
    print("=== Demographic Group Summary ===")
    display(df_grp.head(20))

## 5. 결과 확인 및 산출물 안내

- **ChatGPT 분석용 요약본**: `{output_dir}/CHATGPT_SUMMARY.md`
- **결과 해석 안내서**: `{output_dir}/RFW_CONTINUOUS_CALIBRATION.md`
- **실행 무결성 매니페스트**: `{output_dir}/run_manifest.json`

※ Python 모듈 수정 후 노트북을 재실행할 때는 **커널 재시작(Kernel Restart)**이 필수입니다.